# Clean two years of exports end to end

*Data Analysis in Python · Lesson 3 final exercise*

Every report from lesson 4. on starts from one clean table of the store's sessions, January 2024 to December 2025. Build it from the 24 monthly exports in the program's data store, and make the notebook prove it's clean.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between the parameters cell and the **Check your work** cell. If you're stuck on a step, open its hint under the **Check your work** cell.

1. Read all 24 exports, with dates as dates and revenue as numbers in every file.
2. Create the `clean` table: the 24 exports in one table, with numbered rows and:
   - every lost campaign (`NaN`, `(not set)` or `-`) labeled `"(not set)"`
   - no copies from overlapping exports, no rows dated after `END`, and no test traffic
   - every refund (negative revenue) kept
   - a `source` column with the standardized source (`SOURCE_ALIASES`), next to the export's own `session_source`
3. Create `checks` with a `validate(df, start, end)` function with topic 3.5.'s six checks, and `assert` that they all pass.
4. Create `total_sessions` (the total sessions of the `clean` table) and `net_revenue` (its total revenue, refunds included, rounded to 2 decimals), and check `total_sessions` against `REPORTED_SESSIONS`.
5. Add a text cell that lists what you removed, and how many rows each step took out.
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `clean` table has 43,035 rows from January 1, 2024 to December 31, 2025, with 4,585 refund rows, 4,880 campaigns labeled `(not set)` and a `source` column with 12 sources. `checks` has six checks, all `True`. `total_sessions` is `2506005`, the same as `REPORTED_SESSIONS`, and `net_revenue` is `5078823.67`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]
START, END = "2024-01-01", "2025-12-31"
REPORTED_SESSIONS = 2506005  # the analytics platform's report for 2024 and 2025
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    clean = found.get("clean")
    fingerprints = {
        45760: "the copies from overlapping exports are still in. Remove them with drop_duplicates(), after fixing the types.",
        43219: "March 2025 was read month first: its days 1 to 12 moved to other months, and its copies of March 1 and 2 no longer match February's. Read that file with format=\"%d/%m/%Y\".",
        43266: "revenue was converted after stacking, so the files that already had numbers became NaN, and their copies no longer match. Convert each file's revenue before stacking, only if it's text.",
        42529: "the sources were standardized before removing duplicates: 506 real rows became identical and were removed. Remove duplicates first, and put the standardized values in a new source column.",
        43075: "the 40 rows dated after END are still in. Keep the rows with date <= END.",
        43095: "the 60 test rows are still in. Remove the rows whose session_medium is test.",
        38450: "the 4,585 refund rows are gone. Refunds are real money going back: keep the rows with negative revenue.",
    }
    table_ok = False
    if not isinstance(clean, pd.DataFrame):
        results.append((False, "clean isn't a DataFrame yet. Read the 24 files, stack them, and clean the stack."))
    elif not pd.api.types.is_datetime64_any_dtype(clean["date"]):
        results.append((False, "clean's date column isn't dates. One file has its dates day first: convert each file's dates with the format it uses."))
    elif not pd.api.types.is_numeric_dtype(clean["revenue"]):
        results.append((False, "clean's revenue column isn't numeric. Convert the files with revenue as text, each one before stacking."))
    elif clean["revenue"].isna().any():
        results.append((False, f"{clean['revenue'].isna().sum():,} rows have NaN revenue: " + fingerprints[43266]))
    elif len(clean) in fingerprints:
        results.append((False, f"clean has {len(clean):,} rows: " + fingerprints[len(clean)]))
    elif len(clean) != 43035:
        results.append((False, f"clean has {len(clean):,} rows, but the clean table has 43,035. Check each step's row count: more than one problem may still be in."))
    elif not clean.index.is_unique:
        results.append((False, "clean has the right rows, but its index repeats. Stack with ignore_index=True."))
    elif clean["sessions"].sum() != 2506005 or clean["transactions"].sum() != 65106 or abs(clean["revenue"].sum() - 5078823.67) > 0.005:
        results.append((False, "clean has 43,035 rows, but its totals don't match the clean table. Check that no step changed sessions, transactions or revenue."))
    else:
        table_ok = True
        results.append((True, "clean has the 43,035 real rows: 2,506,005 sessions, 65,106 transactions and $5,078,823.67 in net revenue."))

    if table_ok:
        campaigns = clean["session_campaign"]
        if campaigns.isna().any() or (campaigns == "-").any():
            results.append((False, f"{campaigns.isna().sum() + (campaigns == '-').sum():,} lost campaigns aren't labeled yet. Read the files with na_values for (not set) and -, then fillna(\"(not set)\")."))
        elif (campaigns == "(not set)").sum() == 4880:
            results.append((True, "4,880 rows have their lost campaign labeled (not set)."))
        else:
            results.append((False, f"{(campaigns == '(not set)').sum():,} rows are labeled (not set), but 4,880 lost their campaign."))
        if "source" not in clean.columns:
            results.append((False, "clean has no source column. Add it: the session_source lowercased, stripped and with SOURCE_ALIASES replaced."))
        elif clean["source"].nunique() == 12 and clean["session_source"].nunique() == 21:
            results.append((True, "source has 12 sources, and session_source keeps the export's 21 spellings."))
        elif clean["session_source"].nunique() != 21:
            results.append((False, "clean's session_source has changed. Keep it as the export had it, and put the standardized values in source."))
        else:
            results.append((False, f"source has {clean['source'].nunique()} values, but there are 12 sources. Lowercase, strip, then replace the aliases."))
        if (clean["revenue"] < 0).sum() == 4585:
            results.append((True, "The 4,585 refund rows are in: -$663,844.97 of revenue paid back."))

    checks = found.get("checks")
    if not isinstance(checks, dict) or len(checks) != 6:
        results.append((False, "checks should be validate(clean, START, END), with topic 3.5's six checks."))
    elif not all(bool(value) for value in checks.values()):
        results.append((False, f"Your own checks fail: {[name for name, value in checks.items() if not value]}. Fix the table, not the checks."))
    else:
        results.append((True, "Your six checks pass on clean."))
    if history is not None and not any("assert " in source for source in history):
        results.append((False, "Add an assert after checks, so Run all stops on a bad table: assert all(checks.values()), checks."))

    total_sessions = found.get("total_sessions")
    net_revenue = found.get("net_revenue")
    if isinstance(total_sessions, (pd.DataFrame, pd.Series)):
        results.append((False, f"total_sessions is a {type(total_sessions).__name__}, not a number. Total one column: clean[\"sessions\"].sum()."))
    elif total_sessions == 2506005 and pd.api.types.is_number(net_revenue) and abs(net_revenue - 5078823.67) < 0.005:
        results.append((True, "total_sessions matches the platform's 2,506,005, and net revenue is $5,078,823.67."))
    elif pd.api.types.is_number(net_revenue) and abs(net_revenue - 5742668.64) < 0.005:
        results.append((False, "net_revenue is $5,742,668.64: the refunds aren't subtracted. Net revenue is the total of every row's revenue, negative ones included."))
    else:
        results.append((False, f"total_sessions should be clean[\"sessions\"].sum() and net_revenue round(clean[\"revenue\"].sum(), 2). They are {total_sessions!r} and {net_revenue!r}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. This is the table lesson 4 starts from.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 1: read the 24 exports</b></summary>

Write one function that reads one month (topic 3.2.), and call it for every month in `MONTHS` with a comprehension, as in topic 2.5. Inside the function: `na_values` for the campaign column (topic 3.1.), the date converted with an explicit format, and revenue converted only if it's text (topic 3.2.). One file is day-first: if `format="%Y-%m-%d"` stops with a `ValueError` on one month, that month is the exception, so make a list of the day-first months and choose the format from it.

~~~python
def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": [...]})
    export["date"] = pd.to_datetime(export["date"], format=...)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace(...).str.replace(...).astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
len(raw)
~~~

</details>

<details>
<summary><b>Step 2: clean the table, in the right order</b></summary>

The order matters (topic 3.4.). Label the lost campaigns first (topic 3.1.), so that copies from overlapping exports are equal. Then drop the copies, keep the rows up to `END`, and drop the `test` medium. Add `source` last, as a new column (topic 3.3.): changing `session_source` before `drop_duplicates()` makes some real rows look like copies. Keep each step's table, so you can count what it removed.

~~~python
clean = raw.copy()
clean["session_campaign"] = clean["session_campaign"].fillna(...)
deduped = clean.drop_duplicates()
in_range = deduped[deduped["date"] <= ...]
clean = in_range[in_range["session_medium"] != ...].copy()
clean["source"] = clean["session_source"].str.lower().str.strip().replace(...)
len(raw) - len(deduped), len(deduped) - len(in_range), len(in_range) - len(clean), len(clean)
~~~

</details>

<details>
<summary><b>Step 3: validate the table</b></summary>

Topic 3.5.'s function, unchanged: a dictionary with the six checks' names as keys and `True` or `False` as values. Then one `assert` that stops the notebook if any check fails, with the dictionary as its message.

~~~python
def validate(df, start, end):
    """Return each check's name and whether the table passes it."""
    return {
        "dates in range": bool(df["date"].between(start, end).all()),
        "no duplicate rows": ...,
        "no test traffic": ...,
        "no missing campaigns": ...,
        "numbers are numbers": all(pd.api.types.is_numeric_dtype(df[column]) for column in [...]),
        "transactions within sessions": ...,
    }


checks = validate(clean, START, END)
assert all(checks.values()), checks
~~~

</details>

<details>
<summary><b>Step 4: the totals</b></summary>

Two sums, one of them rounded, and a comparison with the platform's number (topic 3.5.).

~~~python
total_sessions = clean[...].sum()
net_revenue = round(..., 2)
total_sessions == REPORTED_SESSIONS
~~~

</details>

<details>
<summary><b>Step 5: the text cell</b></summary>

Choose **Insert → Text cell**, and use the counts from step 2: how many rows the 24 exports had, how many copies, future rows and test rows each step removed, and how many rows are left.

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
DAY_FIRST_MONTHS = ["2025-03"]  # found when format="%Y-%m-%d" stopped with a ValueError on this file


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
len(raw)
```

```python
clean = raw.copy()
clean["session_campaign"] = clean["session_campaign"].fillna("(not set)")
deduped = clean.drop_duplicates()
in_range = deduped[deduped["date"] <= END]
clean = in_range[in_range["session_medium"] != "test"].copy()
clean["source"] = clean["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
len(raw) - len(deduped), len(deduped) - len(in_range), len(in_range) - len(clean), len(clean)
```

```python
def validate(df, start, end):
    """Return each check's name and whether the table passes it."""
    return {
        "dates in range": bool(df["date"].between(start, end).all()),
        "no duplicate rows": not df.duplicated().any(),
        "no test traffic": not (df["session_medium"] == "test").any(),
        "no missing campaigns": not df["session_campaign"].isna().any(),
        "numbers are numbers": all(pd.api.types.is_numeric_dtype(df[column]) for column in ["sessions", "transactions", "revenue"]),
        "transactions within sessions": bool((df["transactions"] <= df["sessions"]).all()),
    }


checks = validate(clean, START, END)
assert all(checks.values()), checks
checks
```

```python
total_sessions = clean["sessions"].sum()
net_revenue = round(clean["revenue"].sum(), 2)
total_sessions == REPORTED_SESSIONS, total_sessions, net_revenue
```

```markdown
Removed from the 45,868 rows of the 24 exports: 2,733 copies from overlapping exports, 40 rows dated after December 31, 2025, and 60 rows of internal test traffic. 43,035 rows are left, refunds included.
```

</details>